In [ ]:
# some initialization stuff and general imports (see init.py for details)
%run ../init.py
%matplotlib inline
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, f1_score


# Machine Learning Operations (MLOps)
### The Engineering Lifecycle of ML-Powered Software

Interactive lecture slides based on [ml-ops.org](https://ml-ops.org) (CRISP-ML(Q)) and standard principles of statistical decision theory and quality assurance.

- **Why MLOps?** The Three Levels of ML Software (Data, Model, Code)
- **The CRISP-ML(Q) Process Model** (6 Quality-Assured Lifecycle Phases)
- **Phase 1: Business & Data Understanding** (KPIs & Non-ML Baselines)
- **Phase 2: Data Engineering** (Validation, Imputation & DVC Versioning)
- **Phase 3: Model Engineering** (Inductive Bias & Hypotheses)
- **Phase 4: Quality Assurance & Decision Theory** (Confusion Matrix, ROC/AUC, $S$-fold CV)
- **Phase 5: Deployment Patterns** (REST APIs, Embedded & Streaming)
- **Phase 6: Monitoring & Maintenance** (Data Drift, Concept Drift & Continuous Training CT)
- **Hands-On End-to-End Tutorial Lab**


# Why MLOps?

- In traditional software engineering, developers write algorithmic logic that transforms input data into outputs:
  $$\text{Input} + \text{Code} \implies \text{Output}$$
- In machine learning, algorithms discover patterns from data and outputs to generate the program:
  $$\text{Input} + \text{Output} \implies \text{Model}$$
- **The Challenge:** ML applications are not just code — they are a tightly coupled trinity of **Data**, **Model**, and **Code**.

> 📘 **Definition: Machine Learning Operations (MLOps)**
>
> An engineering discipline that unifies ML system development (Dev) with operations (Ops) to standardize, automate, and monitor the continuous delivery of reproducible, testable, and evolvable ML products in production.

<small>📖 *Source: [ml-ops.org](https://ml-ops.org); Continuous Delivery Foundation SIG MLOps*</small>


## The Three Levels of ML-Powered Software

<center>
    <img src="images/mlops_crisp_cycle.svg" width="90%"/>
</center>

- **Data Engineering Pipeline**: Ingestion, validation, cleaning, feature transformation, and data versioning.
- **ML Model Pipeline**: Algorithm selection, distributed training, hyperparameter optimization, and acceptance testing.
- **Application / Software Pipeline**: Packaging (ONNX, Docker), serving endpoints (REST/gRPC), monitoring, and automated retraining triggers.


# The CRISP-ML(Q) Process Model

To overcome the ad-hoc nature of early ML projects, the industry established **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*):

1. **Business & Data Understanding**: Objectives, success KPIs, non-ML heuristic baseline.
2. **Data Engineering**: Exploration, wrangling, labeling, splitting, and versioning.
3. **Machine Learning Model Engineering**: Model selection, inductive bias, training, hyperparameter tuning.
4. **Quality Assurance**: Decision theory, performance metrics, robustness, fairness, $S$-fold validation.
5. **Deployment**: Serving architectures (batch, real-time, edge), canary and shadow rollouts.
6. **Monitoring & Maintenance**: Covariate shift, concept drift, and the Continuous Training (CT) feedback loop.

<small>📖 *Source: [Studer, S., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. Machine Learning and Knowledge Extraction, 3(2), 392–413](https://doi.org/10.3390/make3020020)*</small>


# Business & Data Understanding

- **Establish Clear, Measurable KPIs**:
  - Technical metrics (e.g. Accuracy, AUC) must directly translate into business/scientific KPIs (e.g. reduction in missed cancer lesions, reduction in customer churn, monetary savings).
- **The Non-ML Heuristic Baseline (Crucial Best Practice)**:
  - Before training complex neural networks, always implement a simple non-machine-learning rule-based heuristic or majority-vote baseline.
  - *Why?* A non-ML baseline establishes the minimum performance threshold. If an expensive deep model cannot significantly beat a simple heuristic, the ML project is economically unjustified.
- **Feasibility Assessment & The Machine Learning Canvas**:
  - Are sufficient high-quality training observations available?
  - What are the latency, throughput, privacy, and regulatory constraints?


# Data Engineering & Quality Assurance

- **Data Ingestion & Quality Profiling**:
  - Scan for schema anomalies, missing values, multimodal distributions, and label inconsistencies.
- **Data Wrangling & Imputation**:
  - Impute missing features cleanly (median/mean for numerical, mode for categorical) using pipeline transformers fitted strictly on training data.
- **Data Versioning (DVC — Data Version Control)**:
  - Code is versioned with Git; large datasets cannot be stored directly in Git repositories.
  - Tools like **DVC** compute content hashes (e.g. md5) of dataset files, storing lightweight `.dvc` tracking pointers in Git while syncing large binary artifacts to remote object storage (S3, GCS, MinIO).
- **Dataset Partitioning**:
  - **Training Set** (~70%): Used to optimize model parameters $\mathbf{w}$.
  - **Validation Set** (~15%): Used to tune hyperparameters and choose model architectures.
  - **Test Set** (~15%): Held out until final acceptance testing to measure true generalization.

> ⚠️ **Important:** Never use future data to predict past events (use temporal splits for time-series), and never fit scalers or imputers on the test set!


# Model Engineering & Inductive Bias

> ⚠️ **Important: Inductive Bias Theorem**
>
> *"A learner that makes no prior assumptions regarding the identity of the target concept has no rational basis for classifying any unseen instances."*

<small>📖 — *Tom M. Mitchell (1997), Machine Learning, McGraw-Hill*</small>

- Every learning algorithm incorporates an **inductive bias**:
  - Linear models assume target relationships are hyperplanes.
  - Convolutional networks assume spatial translation equivariance and local connectivity.
  - Recurrent networks assume sequential temporal causality.
- **Model Selection Workflow**:
  - Begin with simple interpretable baselines (Logistic Regression, Decision Trees).
  - Scale up to expressive architectures (Deep Neural Nets, Ensembles) only if validation performance improves beyond variance limits.
- **Model Packaging**:
  - Exporting serialized model artifacts into interoperable standards: **ONNX** (Open Neural Network Exchange), **TorchScript**, or self-contained Docker images.


# Quality Assurance & Decision Theory

How do we evaluate whether a model is safe and effective for production deployment?


## The Confusion Matrix

- For a binary classification problem ($y \in \{0, 1\}$), predictions partition into four outcomes:

$$
\begin{array}{c|c|c}
\text{True / Predicted} & \text{Predicted Positive } (\hat{y}=1) & \text{Predicted Negative } (\hat{y}=0) \\
\hline
\text{Actual Positive } (y=1) & \text{True Positive } (\mathrm{TP}) & \text{False Negative } (\mathrm{FN}) \; \text{[Type II Error]} \\
\hline
\text{Actual Negative } (y=0) & \text{False Positive } (\mathrm{FP}) \; \text{[Type I Error]} & \text{True Negative } (\mathrm{TN}) \\
\end{array}
$$

- **Accuracy** ($A$): Good for balanced data, highly misleading under class imbalance:
  $$A = \frac{\mathrm{TP} + \mathrm{TN}}{\mathrm{TP} + \mathrm{TN} + \mathrm{FP} + \mathrm{FN}}$$
- **Precision** ($P$): When the model predicts positive, how often is it right?
  $$P = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}$$
- **Recall / Sensitivity** ($R$): Out of all true positive cases, how many did the model find?
  $$R = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN}}$$
- **$F_1$-Score**: Harmonic mean balancing precision and recall:
  $$F_1 = 2 \cdot \frac{P \cdot R}{P + R} = \frac{2\mathrm{TP}}{2\mathrm{TP} + \mathrm{FP} + \mathrm{FN}}$$


## Expected Loss & Cost-Sensitive Decisions

- In real-world applications, different mistakes carry radically asymmetric consequences:
  - In medical screening, a **False Negative** (missing a malignant tumor) is potentially fatal.
  - A **False Positive** (unnecessary follow-up biopsy) incurs minor financial/stress cost.
- **The Loss Matrix $\mathbf{L}$**:
  $$L_{kj} = \text{cost of deciding class } j \text{ when the true state is class } k$$
- **Expected Loss Minimization**:
  Choose class $j$ that minimizes expected risk:
  $$\mathbb{E}[L] = \sum_k L_{kj} \, p(C_k \mid \mathbf{x})$$
- **Receiver Operating Characteristic (ROC) & AUC**:
  - Plots **True Positive Rate** (Recall) vs. **False Positive Rate** ($\frac{\mathrm{FP}}{\mathrm{FP}+\mathrm{TN}}$) across all possible decision thresholds $\tau \in [0, 1]$.
  - **AUC (Area Under Curve)** $\in [0.5, 1.0]$: Measures ranking capability independent of the chosen operational threshold.


# Model Deployment & Serving Patterns

How do we serve predictions reliably to end-users and downstream services?

1. **Model-as-a-Service (REST / gRPC Microservice)**:
   - The model is hosted behind a web server (e.g. FastAPI, Triton Inference Server).
   - Client sends JSON feature payload $\implies$ server returns prediction synchronously.
   - *Best for:* Interactive applications, mobile/web backends (latency < 100ms).
2. **Batch Offline Scoring**:
   - Model runs periodically (hourly/nightly cron) over a database table, storing predictions in a cache.
   - *Best for:* High-volume recommendations, periodic risk scores, fraud reports.
3. **Embedded / Edge Model**:
   - Serialized model runs directly inside client applications or IoT edge hardware (CoreML, TFLite).
   - *Best for:* Zero-latency requirements, offline operation, strict privacy.

### Deployment Rollout Strategies
- **Canary Deployment**: Route 5% of traffic to the new model; monitor error rates before scaling to 100%.
- **Blue-Green Deployment**: Run both versions simultaneously; instantly switch router upon verification.
- **Shadow Mode**: New model runs in parallel on live requests, but its outputs are only logged, not shown to users.


# Monitoring, Drift & Continuous Training (CT)

A model in production decays over time because the world is non-stationary.

### The Three Flavors of Production Drift
1. **Covariate Shift (Data Drift)**: The input distribution $p(\mathbf{x})$ shifts over time, but the conditional mapping $p(y \mid \mathbf{x})$ remains constant.
   - *Example*: An imaging sensor changes resolution, or user demographic age shifts.
2. **Concept Drift**: The underlying functional relationship $p(y \mid \mathbf{x})$ changes.
   - *Example*: Consumer buying behaviors shift abruptly during an economic crisis; fraud patterns evolve to bypass model filters.
3. **Prior Probability Shift**: The marginal class frequency $p(y)$ shifts.
   - *Example*: Seasonal flu prevalence spikes in winter.

### The Continuous Training (CT) Automation Loop
- Real-time drift detection (e.g. Kolmogorov-Smirnov statistical tests on feature inputs).
- When drift exceeds threshold $\tau_\mathrm{drift}$ or performance metrics degrade, an automated pipeline triggers: **Data Ingestion $\to$ Retraining $\to$ Validation $\to$ Canary Deployment**.


# Hands-On Lab: The End-to-End MLOps Tutorial

We now walk through a complete, executable Python demonstration of the full MLOps lifecycle:

1. **Data Engineering**: Ingest synthetic loan-default data with imbalanced classes.
2. **Non-ML Heuristic Baseline**: Benchmark a simple threshold heuristic.
3. **Model Engineering**: Train a regularized machine learning classifier.
4. **Quality Assurance**: Compute the Confusion Matrix, Classification Report, and ROC/AUC curve.
5. **Model Serving Simulation**: Implement a mock production REST inference endpoint with payload validation.
6. **Monitoring & Drift Simulation**: Simulate production data drift and trigger an automated Continuous Training alert!


In [ ]:
# 1. Data Engineering: Generate synthetic dataset with realistic class imbalance
np.random.seed(42)

X_raw, y_raw = make_classification(
    n_samples=2000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.85, 0.15], # 15% positive rate (e.g. loan default / fraud)
    random_state=42
)

feature_names = ['credit_score', 'debt_ratio', 'income_stability', 'inquiries_last_6m', 'loan_amount', 'redundant_idx']
df_data = pd.DataFrame(X_raw, columns=feature_names)
df_data['target_default'] = y_raw

# Data Splitting: Stratified 70% Train, 15% Validation, 15% Test
X_train_full, X_test, y_train_full, y_test = train_test_split(df_data[feature_names], df_data['target_default'], test_size=0.15, stratify=df_data['target_default'], random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.176, stratify=y_train_full, random_state=42) # 0.176 * 0.85 approx 0.15

print(f"Dataset summary: Total={len(df_data)}, Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
print(f"Class distribution in training set: Negative={sum(y_train==0)}, Positive={sum(y_train==1)} ({sum(y_train==1)/len(y_train):.1%})")
df_data.head(3)


In [ ]:
# 2. Non-ML Heuristic Baseline: Always predict non-default (majority class)
y_pred_baseline = np.zeros_like(y_val)
base_acc = (y_pred_baseline == y_val).mean()
base_f1 = f1_score(y_val, y_pred_baseline, zero_division=0)

print("=== Non-ML Heuristic Baseline Results ===")
print(f"Heuristic Accuracy : {base_acc:.2%}  (High accuracy due to class imbalance!)")
print(f"Heuristic F1-Score : {base_f1:.2%}  (Completely useless: detects 0% of defaults!)")


In [ ]:
# 3. Model Engineering: Train a cost-sensitive classifier
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Predict probabilities on validation set
y_val_prob = model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_prob >= 0.5).astype(int)

val_f1 = f1_score(y_val, y_val_pred)
print(f"ML Model Validation F1-Score: {val_f1:.2%} (Substantially outperforms heuristic baseline)")


In [ ]:
# 4. Quality Assurance: Confusion Matrix & ROC Curve
cm = confusion_matrix(y_val, y_val_pred)
fpr, tpr, thresholds = roc_curve(y_val, y_val_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Pred Negative', 'Pred Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
axes[0].set_title('Validation Confusion Matrix (Cost-Sensitive)', fontsize=14, fontweight='bold')

# ROC Curve Plot
axes[1].plot(fpr, tpr, color='darkblue', lw=2.5, label=f'ML Model (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Chance (AUC = 0.500)')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Recall / Sensitivity)')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
# 5. Model Serving: Mock REST API endpoint with input schema validation
def mock_predict_endpoint(request_json: dict, deployed_model, features: list, threshold: float = 0.5) -> dict:
    """Simulates a production REST microservice inference endpoint."""
    try:
        # Schema validation
        for feat in features:
            if feat not in request_json:
                return {"status": "error", "message": f"Missing required feature: {feat}"}
        
        # Format feature vector as DataFrame with feature names
        input_df = pd.DataFrame([[request_json[feat] for feat in features]], columns=features)
        prob = float(deployed_model.predict_proba(input_df)[0, 1])
        prediction = int(prob >= threshold)
        
        return {
            "status": "success",
            "prediction": prediction,
            "decision": "REJECT_LOAN" if prediction == 1 else "APPROVE_LOAN",
            "default_probability": round(prob, 4),
            "model_version": "v1.2.0-prod"
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

# Test the serving function with a sample payload
sample_payload = X_test.iloc[0].to_dict()
response = mock_predict_endpoint(sample_payload, model, feature_names)
print("=== Mock Production REST API Response ===")
print(json.dumps(response, indent=2))


In [ ]:
# 6. Monitoring & Maintenance: Simulate Production Covariate Shift (Data Drift)
# In production, incoming applicant debt_ratio and credit scores deteriorate
X_prod_drifted = X_test.copy()
X_prod_drifted['debt_ratio'] += 2.5 # Significant feature shift
X_prod_drifted['credit_score'] -= 1.8

prod_probs = model.predict_proba(X_prod_drifted)[:, 1]
prod_preds = (prod_probs >= 0.5).astype(int)
drifted_f1 = f1_score(y_test, prod_preds)

# Statistical Drift Metric: Population Stability / Mean Shift
drift_magnitude = np.abs(X_prod_drifted['debt_ratio'].mean() - X_train['debt_ratio'].mean())
DRIFT_THRESHOLD = 1.0

print("=== Production Continuous Monitoring (CM) Report ===")
print(f"Original Test F1-Score : {f1_score(y_test, model.predict(X_test)):.2%}")
print(f"Post-Drift F1-Score    : {drifted_f1:.2%} (Degraded performance!)")
print(f"Debt Ratio Mean Shift  : {drift_magnitude:.3f} (Threshold = {DRIFT_THRESHOLD})")

if drift_magnitude > DRIFT_THRESHOLD:
    print("🚨 ALERT: Severe Covariate Shift detected in production stream!")
    print("🔄 TRIGGER: Automated Continuous Training (CT) pipeline initiated.")
    print("   -> Fetching new labeled window...")
    print("   -> Retraining model...")
    print("   -> Running automated acceptance tests...")
    print("   -> Deploying candidate model to Canary stage.")


# Summary & Key Takeaways

- **ML is Not Just Code**: MLOps manages the interconnected lifecycle of **Code + Data + Model**.
- **CRISP-ML(Q) Standard**: Provides a 6-phase quality-assured process model from business KPI formulation to post-deployment monitoring.
- **Decision Theory & Asymmetric Costs**: Accuracy is dangerous under class imbalance; always optimize expected loss, Precision, Recall, and ROC/AUC.
- **Serving Patterns**: Choose between REST microservices (real-time), Batch scoring (offline), and Embedded models (edge).
- **Continuous Monitoring is Mandatory**: All models decay in non-stationary environments. Detect Covariate Shift and Concept Drift early to trigger automated Continuous Training (CT).

<small>📖 *Source: [ml-ops.org](https://ml-ops.org); Studer et al. (2021); Christopher M. Bishop and Hugh Bishop (2024)*</small>
